In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

#################################
# 1. Config & Paths 
#################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_path = "C:/Users/user/Desktop/IDS_masters/dataset/audi_robust_0305.npz"
model1_path = "C:/Users/user/Desktop/IDS_masters/model/TCN1_0308_0535.pth"
model2_path = "C:/Users/user/Desktop/IDS_masters/model/TCN2_0308_0535.pth"

batch_size = 64
Stage1_CH = [0, 1, 2, 3, 4, 5]
Stage2_CH = [4, 9]

num_input1 = 6
num_input2 = 2

In [18]:
#################################
# 2. Dataset Load Class
#################################
class LoadDatset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

#################################
# 3. Causal Convolution Layer
#################################
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )
        self.left_pad = (kernel_size - 1) * dilation

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))
        return super().forward(x)

#################################
# 4. TCN Blocks
#################################
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernal_size=3,  dilation=1, dropout=0.1):
        super().__init__()

        self.conv1 = CausalConv1d(n_inputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.conv3 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        def init_one(layer):
            w = getattr(layer, "weight_orig", None)
            if w is None:
                w = layer.weight
            nn.init.kaiming_normal_(w)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        init_one(self.conv1)
        init_one(self.conv2)
        init_one(self.conv3)

        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight)
            if self.downsample.bias is not None:
                nn.init.zeros_(self.downsample.bias)
                
    def forward(self, x):
        out = self.conv1(x)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = self.conv3(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channel, kernel_size=3, dropout =0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channel)
        dilation = [1,2,4]

        for i in range(num_levels):
            dilation_size = dilation[i]
            in_channels = num_inputs if i == 0 else num_channel[i-1]
            out_channels = num_channel[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,  dilation = dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

#################################
# 5. Model Architecture (SeqIDS)
#################################
class SeqIDS(nn.Module):
    def __init__(self, num_input, num_classes, dropout_rate=0.5):
        super(SeqIDS, self).__init__()

        self.tcn = TemporalConvNet(
            num_inputs=num_input,
            num_channel=[32, 64, 128],
            kernel_size=3,
            dropout=dropout_rate
        )

        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Conv1d(128, num_classes, kernel_size=1)

    def forward(self, x, return_attn=False):
        x = self.tcn(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        if return_attn:
            return logits
        
        return logits

#################################
# 8.  Multi TCN (Evaluation)
#################################

# 모델 초기화 (드롭아웃은 평가 시 무시되지만 구조를 위해 0.2 유지)
model1 = SeqIDS(num_input=num_input1, num_classes=3, dropout_rate=0.5).to(device)
model2 = SeqIDS(num_input=num_input2, num_classes=2, dropout_rate=0.5).to(device)

# 저장된 weight 로드 (보안 경고 방지)
state1 = torch.load(model1_path, map_location=device, weights_only=True)
model1.load_state_dict(state1)

state2 = torch.load(model2_path, map_location=device, weights_only=True)
model2.load_state_dict(state2)

model1.eval()
model2.eval()

# 데이터 로드
test_data = np.load(test_path)
X_np, y_np = test_data["X"], test_data["y"]

test_ds = LoadDatset(X_np, y_np)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False
)

print("⏳ [Step 1] 전체 데이터 모델 예측값 캐싱 중... (1회만 수행)")
all_labels = []
all_p1 = []
all_logits1 = []
all_mapped2 = []

with torch.no_grad():
    for input_batch, labels_batch in test_loader:
        input_batch = input_batch.to(device)
        labels_batch = labels_batch.to(device)

        # Stage 1 연산
        x1 = input_batch[:, Stage1_CH, :]
        logit1 = model1(x1)
        p1 = torch.softmax(logit1, dim=1)

        # Stage 2 연산
        x2 = input_batch[:, Stage2_CH, :]
        logit2 = model2(x2)
        pred2 = logit2.argmax(dim=1)
        
        mapped2 = pred2.clone()
        mapped2[pred2 == 0] = 0 
        mapped2[pred2 == 1] = 4

        # VRAM 메모리 터짐 방지를 위해 CPU로 옮겨서 저장
        all_labels.append(labels_batch.cpu())
        all_p1.append(p1.cpu())
        all_logits1.append(logit1.cpu())
        all_mapped2.append(mapped2.cpu())

# 리스트에 담긴 텐서들을 하나로 병합
all_labels = torch.cat(all_labels, dim=0)
all_p1 = torch.cat(all_p1, dim=0)
all_logits1 = torch.cat(all_logits1, dim=0)
all_mapped2 = torch.cat(all_mapped2, dim=0)


print("\n🔍 [Step 2] Fuzzing 최적의 Threshold 탐색 중...")
best_f1_fuzz = 0.0
best_th_fuzz = 0.5
th_dos = 0.9  # DoS는 완벽하므로 0.9 고정

# 0.1부터 0.9까지 0.1 간격으로 시뮬레이션
for th_fuzz in np.arange(0.1, 1.0, 0.1):
    dos_mask = all_p1[:, 1, :] >= th_dos
    fuzz_mask = all_p1[:, 2, :] >= th_fuzz

    pred = torch.zeros_like(all_labels)

    # 마스킹 로직
    only_dos = dos_mask & ~fuzz_mask
    pred[only_dos] = 1

    only_fuzz = fuzz_mask & ~dos_mask
    pred[only_fuzz] = 2

    both = dos_mask & fuzz_mask
    pred[both] = torch.where(
        all_logits1[:, 1, :][both] >= all_logits1[:, 2, :][both],
        torch.ones_like(pred[both]),
        torch.full_like(pred[both], 2)
    )

    decided = dos_mask | fuzz_mask
    pred[~decided] = all_mapped2[~decided]

    # 초고속 Confusion Matrix 연산
    t_flat = all_labels.flatten()
    p_flat = pred.flatten()

    idx = t_flat * 5 + p_flat
    conf_mat = torch.bincount(idx, minlength=25).reshape(5, 5)

    # Fuzzing(인덱스 2) 성능 계산
    tp = conf_mat[2, 2].item()
    fp = conf_mat[:, 2].sum().item() - tp
    fn = conf_mat[2, :].sum().item() - tp

    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)

    print(f" - th_fuzz: {th_fuzz:.1f} => Fuzzing F1: {f1:.4f} (Prec: {prec:.4f}, Rec: {rec:.4f})")

    # 최고 점수 갱신
    if f1 > best_f1_fuzz:
        best_f1_fuzz = f1
        best_th_fuzz = th_fuzz

print(f"\n✅ 탐색 완료! 최적의 Fuzzing Threshold는 {best_th_fuzz:.1f} 입니다.\n")


print("📊 [Step 3] 최적의 Threshold를 적용한 최종 성능 평가")
# 찾은 최적값으로 최종 판정 수행
dos_mask = all_p1[:, 1, :] >= th_dos
fuzz_mask = all_p1[:, 2, :] >= best_th_fuzz

final_pred = torch.zeros_like(all_labels)
final_pred[dos_mask & ~fuzz_mask] = 1
final_pred[fuzz_mask & ~dos_mask] = 2

both = dos_mask & fuzz_mask
final_pred[both] = torch.where(
    all_logits1[:, 1, :][both] >= all_logits1[:, 2, :][both],
    torch.ones_like(final_pred[both]),
    torch.full_like(final_pred[both], 2)
)

decided = dos_mask | fuzz_mask
final_pred[~decided] = all_mapped2[~decided]

# 최종 평가지표 계산
t_flat = all_labels.flatten()
p_flat = final_pred.flatten()

correct = (t_flat == p_flat).sum().item()
total = t_flat.numel()
accuracy = correct / total

idx = t_flat * 5 + p_flat
conf_mat = torch.bincount(idx, minlength=25).reshape(5, 5)

row_sum = conf_mat.sum(dim=1)
tp = conf_mat.diag()
fp = conf_mat.sum(dim=0) - tp
fn = row_sum - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)

# 화면 출력부
present = row_sum > 0
precision_macro = precision_per_class[present].mean().item()
recall_macro    = recall_per_class[present].mean().item()
f1_macro        = f1_per_class[present].mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro, present only): {precision_macro:.4f}")
print(f"Recall(macro, present only)   : {recall_macro:.4f}")
print(f"F1(macro, present only)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat)

EVAL_CLASSES = [0, 1, 2, 4]
eval_idx = torch.tensor(EVAL_CLASSES)

precision_macro = precision_per_class[eval_idx].mean().item()
recall_macro    = recall_per_class[eval_idx].mean().item()
f1_macro        = f1_per_class[eval_idx].mean().item()

LABEL_NAME = {0:"Normal", 1:"Dos", 2:"Fuzzing", 4:"Spoofing"}

print("\n=== Per-class (Attack) Performance ===")
for i in EVAL_CLASSES:
    total_i = int(row_sum[i].item())
    correct_i = int(tp[i].item())
    acc_i = 100.0 * correct_i / total_i if total_i > 0 else 0.0
    
    prec_i = precision_per_class[i].item()
    rec_i = recall_per_class[i].item()
    f1_i = f1_per_class[i].item()

    print(f"{LABEL_NAME[i]:>10s} : Acc {acc_i:6.2f}% ({correct_i}/{total_i}) | F1: {f1_i:.4f} | Prec: {prec_i:.4f} | Rec: {rec_i:.4f}")

⏳ [Step 1] 전체 데이터 모델 예측값 캐싱 중... (1회만 수행)

🔍 [Step 2] Fuzzing 최적의 Threshold 탐색 중...
 - th_fuzz: 0.1 => Fuzzing F1: 0.6903 (Prec: 0.6037, Rec: 0.8061)
 - th_fuzz: 0.2 => Fuzzing F1: 0.6403 (Prec: 0.6137, Rec: 0.6692)
 - th_fuzz: 0.3 => Fuzzing F1: 0.5905 (Prec: 0.6147, Rec: 0.5681)
 - th_fuzz: 0.4 => Fuzzing F1: 0.5434 (Prec: 0.6129, Rec: 0.4880)
 - th_fuzz: 0.5 => Fuzzing F1: 0.4960 (Prec: 0.6125, Rec: 0.4167)
 - th_fuzz: 0.6 => Fuzzing F1: 0.4516 (Prec: 0.6224, Rec: 0.3544)
 - th_fuzz: 0.7 => Fuzzing F1: 0.3932 (Prec: 0.6393, Rec: 0.2839)
 - th_fuzz: 0.8 => Fuzzing F1: 0.3203 (Prec: 0.6770, Rec: 0.2097)
 - th_fuzz: 0.9 => Fuzzing F1: 0.1702 (Prec: 0.7462, Rec: 0.0961)

✅ 탐색 완료! 최적의 Fuzzing Threshold는 0.1 입니다.

📊 [Step 3] 최적의 Threshold를 적용한 최종 성능 평가
Accuracy : 0.9617
Precision(macro, present only): 0.8703
Recall(macro, present only)   : 0.9214
F1(macro, present only)       : 0.8922
Confusion Matrix:
tensor([[844527,    136,  22455,      0,  12306],
        [     0, 338425,      0,     